# Network telemetry warehouse

**Author:** João Pedro de Moura Lima

A reproducible dimensional-modeling case study connecting network operations to analytical engineering.

## TL;DR

The pipeline turns deterministic synthetic flow telemetry into conformed dimensions, SCD2 device history, two fact tables and decision-ready daily marts. Every published metric is reconciled back to its raw input.

## Context & methods

The portfolio data is fictional by design. The grain of `fact_network_flow` is one observed flow; the grain of `fact_incident` is one incident. Device attributes are resolved as of event time through half-open SCD2 validity intervals.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
root = Path.cwd()
if not (root / 'reports').exists():
    root = root.parent
summary = json.loads((root / 'reports' / 'kpi_summary.json').read_text())
quality = json.loads((root / 'reports' / 'data_quality.json').read_text())
summary

## Data quality

Primary-key uniqueness, referential integrity, numeric bounds, SCD2 current-row rules, interval overlap and raw-to-warehouse row reconciliation are blocking tests.

In [ ]:
pd.DataFrame([quality['reconciliation']]), quality['all_passed'], pd.Series(quality['failure_counts'], name='failures')

## Results

In [ ]:
site_daily = pd.read_csv(root / 'reports' / 'site_daily_kpis.csv')
site_daily.groupby('site_name').agg(traffic_gb=('traffic_gb', 'sum'), p95_latency_ms=('p95_latency_ms', 'mean'), incidents=('incident_count', 'sum')).round(2)

In [ ]:
for name in ['daily_traffic_latency.png', 'site_service_quality.png', 'incident_downtime.png']:
    image = plt.imread(root / 'reports' / 'figures' / name)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.imshow(image)
    ax.axis('off')
    plt.show()

## Takeaways

- The model preserves the device version that was valid when each event occurred.
- Incident-day latency is deliberately higher in the synthetic generator, providing a known analytical signal that the warehouse must preserve.
- Operational thresholds and the service-quality score are illustrative; production use requires service-specific SLOs and calibrated business weights.
- No production telemetry or personally identifiable information is included.